# Exploring LAP: assignment algorithms and LP solvers

[Open in Colab](https://colab.research.google.com/github/gromicho/teaching/blob/main/foundations/optimization/exploring-assignment.ipynb) · [Open in Binder](https://mybinder.org/v2/gh/gromicho/teaching/main?urlpath=tree/foundations/optimization/exploring-assignment.ipynb)

By Joaquim Gromicho. Modernized from the original teaching notebook.

Use one cost matrix to compare SciPy’s assignment algorithm, a Pyomo LP solved by CBC, and the same assignment LP solved by GLPK. All three execute.


In [ ]:
# Use installed packages, install only missing ones, without version pins.
# Load the shared teaching utilities from this checkout or a verified download.
from pathlib import Path
import hashlib
import sys
from urllib.request import urlopen

support_path = next((folder / 'support' for folder in [Path.cwd(), *Path.cwd().parents]
                     if (folder / 'support' / 'teaching_utils.py').is_file()), None)
if support_path is None:
    support_path = Path.cwd() / '.teaching-support'
    support_path.mkdir(exist_ok=True)
    helper = support_path / 'teaching_utils.py'
    expected = 'fbfa41e12709a01548e213abba976dad1e21d0abcfd798a2dfd669706cc152bb'
    if not helper.exists() or hashlib.sha256(helper.read_bytes()).hexdigest() != expected:
        url = 'https://raw.githubusercontent.com/gromicho/teaching/f3ad11b77cae7dd05315d314c7e72ee8516aaa3d/support/teaching_utils.py'
        content = urlopen(url, timeout=45).read()
        if hashlib.sha256(content).hexdigest() != expected:
            raise ValueError('Teaching helper version changed; reopen the current course notebook.')
        helper.write_bytes(content)
sys.path.insert(0, str(support_path))
from teaching_utils import ensure_packages

required_packages = {'numpy': 'numpy', 'pandas': 'pandas', 'scipy': 'scipy', 'pyomo': 'pyomo', 'swiglpk': 'swiglpk'}
ensure_packages(required_packages)


In [ ]:
import numpy as np
import pandas as pd
import pyomo.environ as pyo
from scipy.optimize import linear_sum_assignment
from time import perf_counter
from teaching_utils import install_coin_solvers, solve_checked
install_coin_solvers()


# Meet the abstract model!

In [ ]:
def BipartiteMatching():
    model = pyo.AbstractModel("LAP")    
    model.n = pyo.Param(within=pyo.NonNegativeIntegers,default=0,mutable=True)
    model.nodes = pyo.RangeSet(1,model.n)
    model.cost  = pyo.Param(model.nodes, model.nodes, within=pyo.NonNegativeIntegers,mutable=True)
    model.x     = pyo.Var( model.nodes, model.nodes, within = pyo.NonNegativeReals )

    model.obj = pyo.Objective( rule = lambda model : pyo.quicksum( model.cost[i,j]*model.x[i,j] for i in model.nodes for j in model.nodes ), sense=pyo.minimize )
    model.leave = pyo.Constraint( model.nodes, rule = lambda model, i : pyo.quicksum( model.x[i,j] for j in model.nodes ) == 1 )
    model.enter = pyo.Constraint( model.nodes, rule = lambda model, j : pyo.quicksum( model.x[i,j] for i in model.nodes ) == 1 )
    
    return model

In [ ]:
%time model = BipartiteMatching()

# Generate actual data

The original default of 10,000 items creates 100 million assignment variables. Start with 30; vary this explicitly for scaling experiments. The dense cost matrix already uses quadratic memory, before constructing a model.


In [ ]:
rng=np.random.default_rng(2020)
n=30
C=rng.integers(1,100,size=(n,n))
print(f'{n*n:,} variables; cost array: {C.nbytes:,} bytes')


# Solve and compare

In [ ]:
timings=[]
started=perf_counter()
row_indices,column_indices=linear_sum_assignment(C)
algorithm_value=float(C[row_indices,column_indices].sum())
timings.append(dict(method='SciPy assignment',build_seconds=0.0,solve_seconds=perf_counter()-started,value=algorithm_value))


In [ ]:
%time data = { None: dict( n = {None : n}, cost = { (i+1,j+1) : C[i][j] for i in range(n) for j in range(n) } ) }

In [ ]:
started=perf_counter()
lap=model.create_instance(data)
build_seconds=perf_counter()-started
started=perf_counter()
solve_checked(lap,'cbc')
timings.append(dict(method='CBC / Pyomo LP',build_seconds=build_seconds,solve_seconds=perf_counter()-started,value=pyo.value(lap.obj)))
assert all(abs(pyo.value(lap.x[i,j])-round(pyo.value(lap.x[i,j])))<1e-6 for i in lap.nodes for j in lap.nodes)


## GLPK through its Python bindings
The original lesson used a separately installed GLPK executable. The portable `swiglpk` package exposes the actual GLPK library. The code below builds the same nonnegative assignment LP, with row and column sums equal to one. It is shown here because comparing a modeling layer with a direct solver API is part of this lesson. [Binding documentation](https://github.com/opencobra/swiglpk).


In [ ]:
import swiglpk as glp

def assignment_glpk(cost):
    n=cost.shape[0]
    if cost.shape!=(n,n):
        raise ValueError('This example expects a square cost matrix.')
    started=perf_counter()
    problem=glp.glp_create_prob()
    try:
        glp.glp_set_obj_dir(problem,glp.GLP_MIN)
        glp.glp_add_rows(problem,2*n)
        for i in range(1,2*n+1):
            glp.glp_set_row_bnds(problem,i,glp.GLP_FX,1.0,1.0)
        glp.glp_add_cols(problem,n*n)
        rows=glp.intArray(2*n*n+1);cols=glp.intArray(2*n*n+1);values=glp.doubleArray(2*n*n+1)
        k=1
        for i in range(n):
            for j in range(n):
                col=i*n+j+1
                glp.glp_set_col_bnds(problem,col,glp.GLP_LO,0.0,0.0)
                glp.glp_set_obj_coef(problem,col,float(cost[i,j]))
                for row in (i+1,n+j+1):
                    rows[k]=row;cols[k]=col;values[k]=1.0;k+=1
        glp.glp_load_matrix(problem,2*n*n,rows,cols,values)
        build_seconds=perf_counter()-started
        parameters=glp.glp_smcp();glp.glp_init_smcp(parameters)
        parameters.msg_lev=glp.GLP_MSG_OFF
        started=perf_counter()
        error=glp.glp_simplex(problem,parameters)
        solve_seconds=perf_counter()-started
        if error or glp.glp_get_status(problem)!=glp.GLP_OPT:
            raise RuntimeError('GLPK did not establish optimality.')
        assignment=np.array([glp.glp_get_col_prim(problem,j+1) for j in range(n*n)]).reshape(n,n)
        assert np.allclose(assignment.sum(axis=0),1) and np.allclose(assignment.sum(axis=1),1)
        return float(glp.glp_get_obj_val(problem)),build_seconds,solve_seconds
    finally:
        glp.glp_delete_prob(problem)

value,build,solve=assignment_glpk(C)
timings.append(dict(method='GLPK / direct LP',build_seconds=build,solve_seconds=solve,value=value))
comparison=pd.DataFrame(timings)
assert np.allclose(comparison.value,algorithm_value)
display(comparison)


## Interpret the comparison
The assignment polytope has integral vertices, so an LP can recover an assignment without declaring binary variables. Additional side constraints may destroy that property. Timing differences include different interfaces and model-building costs; repeat on several matrices before generalizing. Equal objective values do not require identical assignments when the optimum is tied.
